# Saluki for synonymous eGFP CDS design

[Notebook](saluki-egfp.ipynb) · [Installation](../docs/installation.md)

Optimize the eGFP CDS for predicted mRNA stability. Keep translation and both UTRs fixed.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
cds = (
    "ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCCTGGTCGAGCTGGACGGCGACGTAAAC"
    "GGCCACAAGTTCAGCGTGTCCGGCGAGGGCGAGGGCGATGCCACCTACGGCAAGCTGACCCTGAAGTTCATC"
    "TGCACCACCGGCAAGCTGCCCGTGCCCTGGCCCACCCTCGTGACCACCCTGACCTACGGCGTGCAGTGCTTC"
    "AGCCGCTACCCCGACCACATGAAGCAGCACGACTTCTTCAAGTCCGCCATGCCCGAAGGCTACGTCCAGGAG"
    "CGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCGCCGAGGTGAAGTTCGAGGGCGACACCCTG"
    "GTGAACCGCATCGAGCTGAAGGGCATCGACTTCAAGGAGGACGGCAACATCCTGGGGCACAAGCTGGAGTAC"
    "AACTACAACAGCCACAACGTCTATATCATGGCCGACAAGCAGAAGAACGGCATCAAGGTGAACTTCAAGATC"
    "CGCCACAACATCGAGGACGGCAGCGTGCAGCTCGCCGACCACTACCAGCAGAACACCCCCATCGGCGACGGC"
    "CCCGTGCTGCTGCCCGACAACCACTACCTGAGCACCCAGTCCGCCCTGAGCAAAGACCCCAACGAGAAGCGC"
    "GATCACATGGTCCTGCTGGAGTTCGTGACCGCCGCCGGGATCACTCTCGGCATGGACGAGCTGTACAAGTAA"
)
utr5 = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC"
utr3 = "GCTGGAGCCTCGGTGGCCTAGCTTCTTGCCCCTTGGGCCTCCCCCCAGCCCCTCCTCCCCTTCCTGCACCCGTACCCCCGTGGTCTTTGAATAAAGTCTGAGTGGGCGGCA"
saluki = load_model(
    "saluki", device=device,
    cds_start=len(utr5), cds_end=len(utr5) + len(cds),
    exon_lengths=[len(utr5) + len(cds) + len(utr3)],
)


def encode_utr(sequence):
    indices = torch.tensor(["ACGT".index(base) for base in sequence])
    return torch.nn.functional.one_hot(indices, num_classes=4)


left, right = encode_utr(utr5), encode_utr(utr3)


class CDSScorer:
    task_names = saluki.task_names
    device = saluki.device

    def score(self, sequences):
        return saluki.score([utr5 + sequence + utr3 for sequence in sequences])

    def score_soft(self, probabilities):
        batch = len(probabilities)
        full_transcripts = torch.cat([
            left.to(probabilities).expand(batch, -1, -1),
            probabilities,
            right.to(probabilities).expand(batch, -1, -1),
        ], dim=1)
        return saluki.score_soft(full_transcripts.clamp(0, 1))


model = CDSScorer()
seeds = [cds]

run = Path("outputs/saluki_egfp_cds")




## 2. Generate candidates



In [2]:
designs = generate_designs(
    model, length=len(cds), initial=seeds, objective="maximize",
    gain_fractions=(0.5, 0.9), output_dir=run / "designs",
    is_CDS_optimized=True, fixed_region=[slice(len(cds) - 3, len(cds))],
)
designs.groupby(["algorithm", "gain_fraction"])["score"].agg(["size", "median", "max"])


size    median       max
algorithm                       gain_fraction                          
adalead                         0.5             100  1.513614  1.662446
                                0.9             100  2.565040  2.609395
basesearch                      0.5             100  1.092009  1.296555
                                0.9             100  1.913996  2.024700
beam_search                     0.5             100  1.562780  1.689252
                                0.9             100  2.678403  2.724082
cross_entropy_method            0.5             100  1.453089  1.590080
                                0.9             100  2.483534  2.522873
den                             0.5             100  1.396423  1.745093
                                0.9             100  2.107789  2.196757
fast_seqprop                    0.5             100  1.283291  1.754693
                                0.9             100  2.215509  2.391140
genetic_algorithm               0.5             100  1.565318  1.828778
                                0.9             100  2.651065  2.722816
ism_greedy                      0.5             100  1.379088  1.530929
                                0.9             100  2.370621  2.487951
ledidi                          0.5             100  0.740454  1.078837
                                0.9             100  1.257882  1.483486
simulated_annealing             0.5             100  0.591725  1.109233
                                0.9             100  0.913594  1.109233
vanilla_activation_maximization 0.5             100  1.501909  1.996902
                                0.9             100  2.254772  2.499013



## 3. Discover motifs

Use the 90% gain stage for motif discovery and panel selection.



In [3]:
designs = designs.loc[designs["gain_fraction"].eq(0.9)].copy()
motifs = call_motifs(designs, model, reverse_complement=False, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-CSGURAGC,1022,627,1594.0
1,2-UCACGCUUAAGU,628,625,1312.0
2,3-GUACCACUGGC,452,452,1014.0
4,9-ACUACGCUCAC,457,435,216.0
5,10-GCAGYGUGCA,14,14,706.0
8,18-AUGGGCACAA,11,11,793.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,ATGGTTAGTAAGGGGGAGGAGCTGTTTACGGGGGTGGTGCCGATCC...,2.724082,1-CSGURAGC;2-UCACGCUUAAGU;3-GUACCACUGGC
573,2,ATGGTTTCGAAGGGGGAGGAGCTGTTTACGGGGGTGGTGCCGATCT...,2.250725,18-AUGGGCACAA
674,3,ATGGTAAGTAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCGATCT...,2.195744,1-CSGURAGC;10-GCAGYGUGCA
579,4,ATGGTTTCGAAGGGGGAGGAGCTCTTCACGGGCGTCGTGCCCATCC...,2.246955,2-UCACGCUUAAGU
696,5,ATGGTTAGCAAGGGGGAGGAGTTGTTTACGGGGGTGGTTCCGATCC...,2.183468,2-UCACGCUUAAGU;10-GCAGYGUGCA;18-AUGGGCACAA
528,6,ATGGTGAGCAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCGATCC...,2.306072,2-UCACGCUUAAGU;3-GUACCACUGGC;9-ACUACGCUCAC;18-...
601,7,ATGGTTTCGAAGGGGGAGGAGCTCTTCACCGGCGTCGTGCCCATCC...,2.229395,2-UCACGCUUAAGU;9-ACUACGCUCAC
596,8,ATGGTAAGTAAGGGGGAAGAGTTGTTTACGGGGGTGGTGCCGATCT...,2.231494,1-CSGURAGC;18-AUGGGCACAA
24,9,ATGGTTTCGAAGGGGGAGGAGTTGTTCACTGGGGTGGTGCCGATCC...,2.692230,1-CSGURAGC;2-UCACGCUUAAGU;3-GUACCACUGGC;9-ACUA...
521,10,ATGGTTAGCAAGGGTGAGGAGTTGTTTACTGGGGTGGTTCCGATCC...,2.329300,1-CSGURAGC;2-UCACGCUUAAGU;10-GCAGYGUGCA
